# LightGBM hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
# Create subsets of the data for different training sizes - chronplogical order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

## Classification

[Parameters](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMClassifier.html)

### 1k

In [4]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [5]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "binary", # Binary classification objective
        "metric": "auc", # Evaluation metric
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 10.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 100), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 10, 100), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-3, 50.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-3, 50.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.5) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else: # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 5) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_1k_parallel.html")


[I 2026-04-25 15:49:54,190] A new study created in memory with name: no-name-ced8b545-8586-4c00-be5a-36565998c76d
[I 2026-04-25 15:49:54,474] Trial 7 finished with value: 0.5712179487179488 and parameters: {'boosting_type': 'goss', 'num_leaves': 21, 'max_depth': 6, 'learning_rate': 0.005375247656768241, 'scale_pos_weight': 4.516906656567825, 'min_split_gain': 0.1103979123847465, 'min_child_weight': 0.1288644627152889, 'min_child_samples': 99, 'colsample_bytree': 0.3670430924546734, 'reg_alpha': 5.530642048994121, 'reg_lambda': 0.01636303917048146, 'colsample_bynode': 0.7896179605623035, 'min_data_per_group': 88, 'max_cat_threshold': 17, 'cat_l2': 0.01808965212360406, 'cat_smooth': 0.03911966843991824, 'max_cat_to_onehot': 17, 'max_bin': 182, 'n_estimators': 181, 'top_rate': 0.13246871236375835, 'other_rate': 0.15335677928729355}. Best is trial 7 with value: 0.5712179487179488.
[I 2026-04-25 15:49:54,490] Trial 0 finished with value: 0.6432829431105292 and parameters: {'boosting_type': 


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:50:15,773] Trial 258 finished with value: 0.6625008612077578 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 16, 'max_depth': 5, 'learning_rate': 0.02016817377789622, 'scale_pos_weight': 4.544572321715227, 'min_split_gain': 6.917616788450517, 'min_child_weight': 0.423095009900473, 'min_child_samples': 53, 'colsample_bytree': 0.5263266196552063, 'reg_alpha': 0.3448684222562268, 'reg_lambda': 1.0539130847833778, 'colsample_bynode': 0.4160948607697176, 'min_data_per_group': 91, 'max_cat_threshold': 9, 'cat_l2': 19.573588723961972, 'cat_smooth': 0.007903259024223624, 'max_cat_to_onehot': 3, 'max_bin': 122, 'n_estimators': 553, 'subsample': 0.6286852991497383, 'subsample_freq': 3}. Best is trial 151 with value: 0.6773341409548306.
[I 2026-04-25 15:50:15,790] Trial 256 finished with value: 0.6634348218830978 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 16, 'max_depth': 5, 'learning_rate': 0.018923075141209673, 'scale_pos_weight': 5.287978746234686, 'min_sp


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6773
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 19,
    "max_depth": 6,
    "learning_rate": 0.013118621303382356,
    "scale_pos_weight": 4.234357177755502,
    "min_split_gain": 4.150101489297818,
    "min_child_weight": 0.1439268407905771,
    "min_child_samples": 47,
    "colsample_bytree": 0.6376678823246574,
    "reg_alpha": 0.29085902006459885,
    "reg_lambda": 0.3198616601646805,
    "colsample_bynode": 0.4286302115954977,
    "min_data_per_group": 97,
    "max_cat_threshold": 8,
    "cat_l2": 30.200602897458936,
    "cat_smooth": 0.014021044837121193,
    "max_cat_to_onehot": 6,
    "max_bin": 166,
    "n_estimators": 372,
    "subsample": 0.6440572461920292,
    "subsample_freq": 4,
}

--- PARAMETER IMPORTANCE ---
  min_child_samples   : 0.6012
  min_data_per_group  : 0.1105
  reg_alpha        

In [6]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_lgbm.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"

print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print(f"Optuna Val AUC: {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 19, 'max_depth': 6, 'learning_rate': 0.013118621303382356, 'scale_pos_weight': 4.234357177755502, 'min_split_gain': 4.150101489297818, 'min_child_weight': 0.1439268407905771, 'min_child_samples': 47, 'colsample_bytree': 0.6376678823246574, 'reg_alpha': 0.29085902006459885, 'reg_lambda': 0.3198616601646805, 'colsample_bynode': 0.4286302115954977, 'min_data_per_group': 97, 'max_cat_threshold': 8, 'cat_l2': 30.200602897458936, 'cat_smooth': 0.014021044837121193, 'max_cat_to_onehot': 6, 'max_bin': 166, 'n_estimators': 372, 'subsample': 0.6440572461920292, 'subsample_freq': 4, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
Optuna Val AUC: 0.6773
Holdout Test AUC: 0.6715


### 10k

In [7]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "binary", # Binary classification objective
        "metric": "auc", # Evaluation metric
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 10.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 100), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 10, 100), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-3, 50.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-3, 50.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.5) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else: # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 5) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_10k_parallel.html")


[I 2026-04-25 15:50:20,123] A new study created in memory with name: no-name-0e1b946f-0fcd-465b-90dd-4085697b06e7
[I 2026-04-25 15:50:21,299] Trial 3 finished with value: 0.6913162253492493 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 11, 'max_depth': 2, 'learning_rate': 0.011034963897645229, 'scale_pos_weight': 4.7512119761172835, 'min_split_gain': 1.2168498367172176, 'min_child_weight': 0.049842007916317196, 'min_child_samples': 46, 'colsample_bytree': 0.38934040169873113, 'reg_alpha': 0.028008223591814315, 'reg_lambda': 0.004546462719546609, 'colsample_bynode': 0.36914840908335655, 'min_data_per_group': 86, 'max_cat_threshold': 28, 'cat_l2': 0.11929008623773828, 'cat_smooth': 2.7045166632838926, 'max_cat_to_onehot': 17, 'max_bin': 209, 'n_estimators': 256, 'subsample': 0.7061780324838509, 'subsample_freq': 5}. Best is trial 3 with value: 0.6913162253492493.
[I 2026-04-25 15:50:22,208] Trial 0 finished with value: 0.6928118688488882 and parameters: {'boosting_type': 'goss'


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:44,759] Trial 346 finished with value: 0.682481314060926 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.031089305715482334, 'scale_pos_weight': 2.8291571000292732, 'min_split_gain': 8.37642098786528, 'min_child_weight': 0.6224545031234456, 'min_child_samples': 66, 'colsample_bytree': 0.35801281225153686, 'reg_alpha': 0.005527471257538076, 'reg_lambda': 1.8316311920817794, 'colsample_bynode': 0.3545273470066443, 'min_data_per_group': 19, 'max_cat_threshold': 2, 'cat_l2': 0.1253427257535439, 'cat_smooth': 0.20252741972869, 'max_cat_to_onehot': 1, 'max_bin': 63, 'n_estimators': 506, 'top_rate': 0.08684658724254779, 'other_rate': 0.11885620906731485}. Best is trial 245 with value: 0.7013105762499531.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,107] Trial 347 finished with value: 0.697677783203694 and parameters: {'boosting_type': 'goss', 'num_leaves': 24, 'max_depth': 8, 'learning_rate': 0.008357192583382083, 'scale_pos_weight': 6.4781357387833625, 'min_split_gain': 7.72269625891477, 'min_child_weight': 0.2563974317964869, 'min_child_samples': 65, 'colsample_bytree': 0.3606242453223351, 'reg_alpha': 0.005365247992282698, 'reg_lambda': 1.24109993466758, 'colsample_bynode': 0.35723141410217407, 'min_data_per_group': 19, 'max_cat_threshold': 1, 'cat_l2': 0.23013810472812773, 'cat_smooth': 0.5217710399079403, 'max_cat_to_onehot': 1, 'max_bin': 63, 'n_estimators': 536, 'top_rate': 0.06297931569529822, 'other_rate': 0.11437443571151815}. Best is trial 245 with value: 0.7013105762499531.
[I 2026-04-25 15:52:45,144] Trial 344 finished with value: 0.6981534146556354 and parameters: {'boosting_type': 'goss', 'num_leaves': 24, 'max_depth': 8, 'learning_rate': 0.009585359741586238, 'scale_pos_weight': 6.1355443499


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,488] Trial 348 finished with value: 0.6951251266227872 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.008344798047580902, 'scale_pos_weight': 6.378948698590474, 'min_split_gain': 7.798742266163628, 'min_child_weight': 0.24289641442745302, 'min_child_samples': 65, 'colsample_bytree': 0.3563903686308207, 'reg_alpha': 0.005256599222385073, 'reg_lambda': 1.3080771053619846, 'colsample_bynode': 0.3522707366869879, 'min_data_per_group': 19, 'max_cat_threshold': 1, 'cat_l2': 1.6207527773687507, 'cat_smooth': 0.3614211084854265, 'max_cat_to_onehot': 1, 'max_bin': 75, 'n_estimators': 525, 'top_rate': 0.060240795540791356, 'other_rate': 0.6884066274386166}. Best is trial 245 with value: 0.7013105762499531.
[I 2026-04-25 15:52:45,652] Trial 349 finished with value: 0.6979947830211344 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.00843648600337008, 'scale_pos_weight': 6.386785190


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,930] Trial 350 finished with value: 0.6953845159267183 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.008313886763585104, 'scale_pos_weight': 6.8374203652040695, 'min_split_gain': 8.776620444737008, 'min_child_weight': 0.6435010292472431, 'min_child_samples': 65, 'colsample_bytree': 0.3586630388729784, 'reg_alpha': 0.00561221722672104, 'reg_lambda': 1.2279248109170238, 'colsample_bynode': 0.34017770617817794, 'min_data_per_group': 16, 'max_cat_threshold': 1, 'cat_l2': 0.20908455050213204, 'cat_smooth': 0.36723714051701, 'max_cat_to_onehot': 1, 'max_bin': 76, 'n_estimators': 527, 'top_rate': 0.050920908834292025, 'other_rate': 0.6136774896006213}. Best is trial 245 with value: 0.7013105762499531.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7013
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 26,
    "max_depth": 8,
    "learning_rate": 0.011490524171318373,
    "scale_pos_weight": 2.5548028672067264,
    "min_split_gain": 8.402184412779064,
    "min_child_weight": 0.7889913973072552,
    "min_child_samples": 67,
    "colsample_bytree": 0.3322094184323836,
    "reg_alpha": 0.009195021998688829,
    "reg_lambda": 1.241109986317591,
    "colsample_bynode": 0.3432164431126759,
    "min_data_per_group": 13,
    "max_cat_threshold": 3,
    "cat_l2": 4.82194592281236,
    "cat_smooth": 0.41553169506659904,
    "max_cat_to_onehot": 1,
    "max_bin": 66,
    "n_estimators": 511,
    "top_rate": 0.07974625682135272,
    "other_rate": 0.15560637534232696,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.5685
  num_leaves          : 0.0788
  colsample_bytree    : 0.0543
  max_cat_to_onehot   : 0.0503
  min_split_g

In [8]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] GOSS: Trees 511 -> 1022, LR 0.0115 -> 0.0057
BEST PARAMS: {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 8, 'learning_rate': 0.0057452620856591865, 'scale_pos_weight': 2.5548028672067264, 'min_split_gain': 8.402184412779064, 'min_child_weight': 0.7889913973072552, 'min_child_samples': 67, 'colsample_bytree': 0.3322094184323836, 'reg_alpha': 0.009195021998688829, 'reg_lambda': 1.241109986317591, 'colsample_bynode': 0.3432164431126759, 'min_data_per_group': 13, 'max_cat_threshold': 3, 'cat_l2': 4.82194592281236, 'cat_smooth': 0.41553169506659904, 'max_cat_to_onehot': 1, 'max_bin': 66, 'n_estimators': 1022, 'top_rate': 0.07974625682135272, 'other_rate': 0.15560637534232696, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}

Optuna Val AUC:   0.7013
Holdout Test AUC: 0.6986


### 100k

In [9]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512), # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20), # Max tree depth
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "binary", # Binary classification objective
        "metric": "auc", # Evaluation metric
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 30.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 100.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 100.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 1, 1000), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-8, 100.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-8, 100.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }
    
    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.8) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 10) # Frequency of subsampling
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
) 
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_100k_parallel.html")


[I 2026-04-25 15:52:54,717] A new study created in memory with name: no-name-f9d52ece-027d-4720-8100-fbe0f73c5d4e
[I 2026-04-25 15:53:15,646] Trial 4 finished with value: 0.7039879129522958 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 41, 'max_depth': 6, 'learning_rate': 0.07382190803057405, 'scale_pos_weight': 3.529092898526666, 'min_split_gain': 23.190103058564844, 'min_child_weight': 0.012876210208074489, 'min_child_samples': 305, 'colsample_bytree': 0.9431537409012714, 'reg_alpha': 5.698812673178861e-07, 'reg_lambda': 4.614241724882315e-05, 'colsample_bynode': 0.25704899552163574, 'min_data_per_group': 620, 'max_cat_threshold': 99, 'cat_l2': 4.50744967453117e-06, 'cat_smooth': 0.20703022656484735, 'max_cat_to_onehot': 27, 'max_bin': 385, 'n_estimators': 362, 'subsample': 0.9579037649496904, 'subsample_freq': 5}. Best is trial 4 with value: 0.7039879129522958.
[I 2026-04-25 15:53:19,907] Trial 1 finished with value: 0.6978547249787836 and parameters: {'boosting_type': 'gb


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:18,808] Trial 420 finished with value: 0.7090748067860579 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 402, 'max_depth': 19, 'learning_rate': 0.012064509498672367, 'scale_pos_weight': 6.000820536995601, 'min_split_gain': 14.744267543389505, 'min_child_weight': 0.00042452833799367577, 'min_child_samples': 231, 'colsample_bytree': 0.27082014638516005, 'reg_alpha': 0.31773748021833764, 'reg_lambda': 0.28807620257771754, 'colsample_bynode': 0.2544397541629257, 'min_data_per_group': 873, 'max_cat_threshold': 851, 'cat_l2': 0.00010428825682054862, 'cat_smooth': 6.354523678399262e-05, 'max_cat_to_onehot': 42, 'max_bin': 262, 'n_estimators': 917, 'subsample': 0.8070010172623305, 'subsample_freq': 4}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:23,527] Trial 414 finished with value: 0.7124159758925691 and parameters: {'boosting_type': 'goss', 'num_leaves': 407, 'max_depth': 19, 'learning_rate': 0.011853753666117186, 'scale_pos_weight': 2.006428308993827, 'min_split_gain': 2.61457185990671, 'min_child_weight': 0.0014665052132248712, 'min_child_samples': 249, 'colsample_bytree': 0.3453705061879384, 'reg_alpha': 0.29051633589300185, 'reg_lambda': 0.1764762783088752, 'colsample_bynode': 0.2517626085576692, 'min_data_per_group': 841, 'max_cat_threshold': 846, 'cat_l2': 0.00011517358561435425, 'cat_smooth': 2.3811404426187146e-05, 'max_cat_to_onehot': 38, 'max_bin': 171, 'n_estimators': 982, 'top_rate': 0.12642323132058134, 'other_rate': 0.432581154769897}. Best is trial 313 with value: 0.7133944066754897.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:39,232] Trial 415 finished with value: 0.7023183054340401 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 404, 'max_depth': 19, 'learning_rate': 0.04994425792630373, 'scale_pos_weight': 6.129854694140937, 'min_split_gain': 1.38604714656651, 'min_child_weight': 0.00038079338817436617, 'min_child_samples': 247, 'colsample_bytree': 0.25828743719334085, 'reg_alpha': 0.3221174318459293, 'reg_lambda': 0.19585247883772386, 'colsample_bynode': 0.25358630061866566, 'min_data_per_group': 907, 'max_cat_threshold': 848, 'cat_l2': 0.00011947582860669784, 'cat_smooth': 8.069251302818221e-05, 'max_cat_to_onehot': 42, 'max_bin': 205, 'n_estimators': 959, 'subsample': 0.9125659664217687, 'subsample_freq': 2}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:44:05,346] Trial 419 finished with value: 0.7121240496180189 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 403, 'max_depth': 19, 'learning_rate': 0.01187057534666296, 'scale_pos_weight': 5.882966767230629, 'min_split_gain': 1.1314509698807735, 'min_child_weight': 0.00043851962596906565, 'min_child_samples': 248, 'colsample_bytree': 0.2751185538343298, 'reg_alpha': 0.31304603766252603, 'reg_lambda': 0.23570197045556773, 'colsample_bynode': 0.25570255457345836, 'min_data_per_group': 836, 'max_cat_threshold': 320, 'cat_l2': 0.00011404091336086746, 'cat_smooth': 8.632701913644228e-05, 'max_cat_to_onehot': 42, 'max_bin': 201, 'n_estimators': 970, 'subsample': 0.9104826683186029, 'subsample_freq': 4}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7134
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 444,
    "max_depth": 15,
    "learning_rate": 0.013490712742580084,
    "scale_pos_weight": 2.529589934768151,
    "min_split_gain": 3.4510864669147536,
    "min_child_weight": 0.19239271728859866,
    "min_child_samples": 234,
    "colsample_bytree": 0.31324120666189326,
    "reg_alpha": 0.6545836243558268,
    "reg_lambda": 0.1069421692665621,
    "colsample_bynode": 0.21348019642019067,
    "min_data_per_group": 803,
    "max_cat_threshold": 819,
    "cat_l2": 0.00011204200863439848,
    "cat_smooth": 1.5621591746591863,
    "max_cat_to_onehot": 36,
    "max_bin": 258,
    "n_estimators": 872,
    "top_rate": 0.24249410683009498,
    "other_rate": 0.42369427528780756,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.2649
  boosting_type       : 0.1363
  reg_alpha           : 0.1359
  n_estimators        : 0.102

In [10]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")


# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] GOSS: Trees 872 -> 8720, LR 0.0135 -> 0.0013
BEST PARAMS: {'boosting_type': 'goss', 'num_leaves': 444, 'max_depth': 15, 'learning_rate': 0.0013490712742580085, 'scale_pos_weight': 2.529589934768151, 'min_split_gain': 3.4510864669147536, 'min_child_weight': 0.19239271728859866, 'min_child_samples': 234, 'colsample_bytree': 0.31324120666189326, 'reg_alpha': 0.6545836243558268, 'reg_lambda': 0.1069421692665621, 'colsample_bynode': 0.21348019642019067, 'min_data_per_group': 803, 'max_cat_threshold': 819, 'cat_l2': 0.00011204200863439848, 'cat_smooth': 1.5621591746591863, 'max_cat_to_onehot': 36, 'max_bin': 258, 'n_estimators': 8720, 'top_rate': 0.24249410683009498, 'other_rate': 0.42369427528780756, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}

Optuna Val AUC:   0.7134
Holdout Test AUC: 0.7169


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512), # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20), # Max tree depth
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "binary", # Binary classification objective
        "metric": "auc", # Evaluation metric
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2, 10), # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 30.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 100.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 100.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 1, 1000), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-8, 100.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-8, 100.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }
    
    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.8) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 10) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
) 
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Classification/optuna_lgbm_full_parallel.html")


[I 2026-04-25 16:46:50,843] A new study created in memory with name: no-name-b5a904b2-cce6-4933-9fef-b41a3dec45ea


[I 2026-04-25 16:48:26,885] Trial 7 finished with value: 0.7313784756171374 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 470, 'max_depth': 2, 'learning_rate': 0.06864542579365267, 'scale_pos_weight': 7.762779177414871, 'min_split_gain': 2.8621739571144413, 'min_child_weight': 0.0006621577859487494, 'min_child_samples': 205, 'colsample_bytree': 0.41640636205924486, 'reg_alpha': 3.8999094146303785, 'reg_lambda': 2.9664304014079545e-05, 'colsample_bynode': 0.9760590838044547, 'min_data_per_group': 532, 'max_cat_threshold': 383, 'cat_l2': 3.0170725715281244e-07, 'cat_smooth': 1.1646355604702144e-08, 'max_cat_to_onehot': 35, 'max_bin': 66, 'n_estimators': 227, 'subsample': 0.6425236906943967, 'subsample_freq': 2}. Best is trial 7 with value: 0.7313784756171374.
[I 2026-04-25 16:49:57,647] Trial 2 finished with value: 0.7342846827787698 and parameters: {'boosting_type': 'goss', 'num_leaves': 96, 'max_depth': 15, 'learning_rate': 0.11901157219045892, 'scale_pos_weight': 9.791532150

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")


# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298


## Regression

[Parameters](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMRegressor.html)

### 1k

In [ ]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "regression", # Regression objective
        "metric": "rmse", # Evaluation metric
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 10.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 100), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 10, 100), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-3, 50.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-3, 50.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.5) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else: # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 5) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_1k_parallel.html")


[I 2026-04-23 23:16:26,791] A new study created in memory with name: no-name-3d675b1a-14b0-4e81-a71a-4a2cff1ea028
[I 2026-04-23 23:16:27,015] Trial 7 finished with value: 0.3229392463206594 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 14, 'max_depth': 2, 'learning_rate': 0.00632300979343126, 'min_split_gain': 3.0700028144971414, 'min_child_weight': 0.6450042461254675, 'min_child_samples': 59, 'colsample_bytree': 0.6633887636563797, 'reg_alpha': 0.023918958499886, 'reg_lambda': 0.062024377642462325, 'colsample_bynode': 0.6096533399006528, 'min_data_per_group': 38, 'max_cat_threshold': 17, 'cat_l2': 6.736955838140262, 'cat_smooth': 0.01857030454653203, 'max_cat_to_onehot': 20, 'max_bin': 208, 'n_estimators': 177, 'subsample': 0.9020837844222164, 'subsample_freq': 2}. Best is trial 7 with value: 0.3229392463206594.
[I 2026-04-23 23:16:27,027] Trial 1 finished with value: 0.3271874940878877 and parameters: {'boosting_type': 'dart', 'num_leaves': 31, 'max_depth': 4, 'learning_rat


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:16:44,135] Trial 237 finished with value: 0.31928155449193457 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 19, 'max_depth': 8, 'learning_rate': 0.00823347837539408, 'min_split_gain': 0.7562953466990994, 'min_child_weight': 0.0013563009062005985, 'min_child_samples': 33, 'colsample_bytree': 0.6211329843410311, 'reg_alpha': 0.11582667229934947, 'reg_lambda': 1.0270296629959879, 'colsample_bynode': 0.584618686623982, 'min_data_per_group': 96, 'max_cat_threshold': 5, 'cat_l2': 0.0038784778979972815, 'cat_smooth': 0.005719331556156902, 'max_cat_to_onehot': 19, 'max_bin': 196, 'n_estimators': 219, 'subsample': 0.7087137467828426, 'subsample_freq': 1}. Best is trial 131 with value: 0.3156638147881459.
[I 2026-04-23 23:16:44,139] Trial 235 finished with value: 0.31957116002102537 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 19, 'max_depth': 8, 'learning_rate': 0.008336283834706373, 'min_split_gain': 0.792597698707032, 'min_child_weight': 0.003608264489290


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.3157
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 11,
    "max_depth": 2,
    "learning_rate": 0.07937213306760936,
    "min_split_gain": 0.5472069155545647,
    "min_child_weight": 3.5680367770996537,
    "min_child_samples": 16,
    "colsample_bytree": 0.5475645488622791,
    "reg_alpha": 0.16934625427729877,
    "reg_lambda": 0.008991329512538573,
    "colsample_bynode": 0.5058659191501015,
    "min_data_per_group": 83,
    "max_cat_threshold": 9,
    "cat_l2": 0.0025325289393764152,
    "cat_smooth": 0.0465219035655617,
    "max_cat_to_onehot": 4,
    "max_bin": 190,
    "n_estimators": 627,
    "subsample": 0.5584123147455704,
    "subsample_freq": 2,
}

--- PARAMETER I

In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_lgbm.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print(f"Optuna Val RMSE: {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

Optuna Val RMSE: 0.3157
Holdout Test RMSE: 0.3370


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 3, 31), # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "regression", # Regression objective
        "metric": "rmse", # Evaluation metric
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 10.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 5.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 15, 100), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.8), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 10, 100), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 32), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-3, 50.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-3, 50.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 20), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }

    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.5) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else: # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 5) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_10k_parallel.html")


[I 2026-04-23 23:16:47,643] A new study created in memory with name: no-name-4445f17c-d3f0-4a72-a3f3-4c3dd65cdf23
[I 2026-04-23 23:16:48,214] Trial 3 finished with value: 0.3005803681127669 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 5, 'max_depth': 4, 'learning_rate': 0.009907708416442717, 'min_split_gain': 7.917883621653012, 'min_child_weight': 0.019481881617907693, 'min_child_samples': 74, 'colsample_bytree': 0.4864205891464338, 'reg_alpha': 0.1832890201679671, 'reg_lambda': 0.004792999370254282, 'colsample_bynode': 0.3582405669359461, 'min_data_per_group': 98, 'max_cat_threshold': 6, 'cat_l2': 0.3128070470606007, 'cat_smooth': 0.01751630512879628, 'max_cat_to_onehot': 19, 'max_bin': 231, 'n_estimators': 177, 'subsample': 0.6173940184438262, 'subsample_freq': 1}. Best is trial 3 with value: 0.3005803681127669.
[I 2026-04-23 23:16:48,314] Trial 1 finished with value: 0.3005803681127669 and parameters: {'boosting_type': 'goss', 'num_leaves': 24, 'max_depth': 6, 'learning_r


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:19:22,861] Trial 350 finished with value: 0.29743215425675473 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 3, 'learning_rate': 0.010820992397998162, 'min_split_gain': 0.6794970006868866, 'min_child_weight': 0.003574826540555678, 'min_child_samples': 34, 'colsample_bytree': 0.31924749416276005, 'reg_alpha': 0.13357107204886395, 'reg_lambda': 2.6396142459221976, 'colsample_bynode': 0.3099034758104949, 'min_data_per_group': 20, 'max_cat_threshold': 31, 'cat_l2': 0.008608911889004426, 'cat_smooth': 15.486772571645103, 'max_cat_to_onehot': 10, 'max_bin': 70, 'n_estimators': 948, 'top_rate': 0.1785800537932592, 'other_rate': 0.07419732208549432}. Best is trial 251 with value: 0.2970415538303074.
[I 2026-04-23 23:19:23,848] Trial 352 finished with value: 0.2971592763629771 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 3, 'learning_rate': 0.010709849974899316, 'min_split_gain': 0.6589406162880687, 'min_child_weight': 0.003


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:19:24,256] Trial 353 finished with value: 0.2972463316877373 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 3, 'learning_rate': 0.012386847761875737, 'min_split_gain': 0.6397324533060754, 'min_child_weight': 0.003349194409822594, 'min_child_samples': 96, 'colsample_bytree': 0.3000541159993746, 'reg_alpha': 0.21197597118627134, 'reg_lambda': 2.6841434209927217, 'colsample_bynode': 0.33659920546480043, 'min_data_per_group': 61, 'max_cat_threshold': 31, 'cat_l2': 0.017383680223144463, 'cat_smooth': 16.838729282565566, 'max_cat_to_onehot': 10, 'max_bin': 104, 'n_estimators': 792, 'top_rate': 0.17608055231180275, 'other_rate': 0.06971272862805789}. Best is trial 251 with value: 0.2970415538303074.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:19:26,334] Trial 356 finished with value: 0.29762352902448486 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 4, 'learning_rate': 0.010763912127933027, 'min_split_gain': 0.6083490116798661, 'min_child_weight': 0.003295238401044823, 'min_child_samples': 35, 'colsample_bytree': 0.32163388342925603, 'reg_alpha': 0.3013987850414574, 'reg_lambda': 3.959422281713374, 'colsample_bynode': 0.31639602134073314, 'min_data_per_group': 65, 'max_cat_threshold': 11, 'cat_l2': 0.006060185122311754, 'cat_smooth': 16.77557617115421, 'max_cat_to_onehot': 10, 'max_bin': 74, 'n_estimators': 790, 'top_rate': 0.1900199477082122, 'other_rate': 0.057871114956615684}. Best is trial 251 with value: 0.2970415538303074.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:19:26,566] Trial 357 finished with value: 0.29720368464181257 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 3, 'learning_rate': 0.010749395628653916, 'min_split_gain': 0.6237353098993983, 'min_child_weight': 0.0032320955412945083, 'min_child_samples': 98, 'colsample_bytree': 0.3224273841268828, 'reg_alpha': 0.17298905320242539, 'reg_lambda': 2.614515229289757, 'colsample_bynode': 0.31329281426473427, 'min_data_per_group': 13, 'max_cat_threshold': 31, 'cat_l2': 0.006026252209781404, 'cat_smooth': 15.07099313817365, 'max_cat_to_onehot': 10, 'max_bin': 97, 'n_estimators': 792, 'top_rate': 0.10994428142563686, 'other_rate': 0.05654561047885907}. Best is trial 251 with value: 0.2970415538303074.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2970
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 23,
    "max_depth": 3,
    "learning_rate": 0.011794426191191655,
    "min_split_gain": 0.40512543940281964,
    "min_child_weight": 0.003806454892315437,
    "min_child_samples": 22,
    "colsample_bytree": 0.3093169006974718,
    "reg_alpha": 3.338923175079243,
    "reg_lambda": 3.8058108506250505,
    "colsample_bynode": 0.5037234813540757,
    "min_data_per_group": 56,
    "max_cat_threshold": 32,
    "cat_l2": 0.014697101959104104,
    "cat_smooth": 28.805098200037513,
    "max_cat_to_onehot": 14,
    "max_bin": 80,
    "n_estimators": 701,
    "top_rate": 0.19147368678397228,
    "other_rate": 0.15778184295240402,
}

--- PARAMETER IMPORTANCE ---
  colsample_bytree    : 0.5739
  min_split_gain      : 0.1713
  learning_rate       : 0.0878
  boosting_type       : 0.0694
  max_cat_threshold   : 0.0373
  max_bin         

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")


# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GOSS: Trees 701 -> 1402, LR 0.0118 -> 0.0059

Optuna Val RMSE:   0.2970
Holdout Test RMSE: 0.3120


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb
# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512), # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20), # Max tree depth
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "regression", # Regression objective
        "metric": "rmse", # Evaluation metric
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 30.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 100.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 100.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 1, 1000), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-8, 100.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-8, 100.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }
    
    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.8) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 10) # Frequency of subsampling
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
) 
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_100k_parallel.html")


[I 2026-04-23 23:19:35,861] A new study created in memory with name: no-name-2cb1f803-1e69-4f5c-ae93-5746dc805536
[I 2026-04-23 23:19:46,599] Trial 5 finished with value: 0.30712532422948396 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 244, 'max_depth': 13, 'learning_rate': 0.005825308303845486, 'min_split_gain': 10.682341345083838, 'min_child_weight': 0.09745579911706942, 'min_child_samples': 249, 'colsample_bytree': 0.20314825381145704, 'reg_alpha': 0.8400870449518681, 'reg_lambda': 0.0007733483076610174, 'colsample_bynode': 0.4195394816425778, 'min_data_per_group': 447, 'max_cat_threshold': 116, 'cat_l2': 0.0008812355897842124, 'cat_smooth': 2.288590854097493e-07, 'max_cat_to_onehot': 16, 'max_bin': 475, 'n_estimators': 837, 'subsample': 0.2832319785596212, 'subsample_freq': 6}. Best is trial 5 with value: 0.30712532422948396.
[I 2026-04-23 23:19:47,813] Trial 4 finished with value: 0.3052901990090897 and parameters: {'boosting_type': 'goss', 'num_leaves': 500, 'max_depth


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:59:14,778] Trial 377 finished with value: 0.2996406830665806 and parameters: {'boosting_type': 'goss', 'num_leaves': 95, 'max_depth': 7, 'learning_rate': 0.023399219479560928, 'min_split_gain': 0.02755527097742347, 'min_child_weight': 0.022443816481623854, 'min_child_samples': 328, 'colsample_bytree': 0.5479634262724268, 'reg_alpha': 2.9894354954566556e-06, 'reg_lambda': 0.009017068187239465, 'colsample_bynode': 0.7130044138098913, 'min_data_per_group': 41, 'max_cat_threshold': 241, 'cat_l2': 61.63814496429384, 'cat_smooth': 0.0002181733575113688, 'max_cat_to_onehot': 4, 'max_bin': 380, 'n_estimators': 357, 'top_rate': 0.49367106159416435, 'other_rate': 0.43549742779477657}. Best is trial 277 with value: 0.29933817748389.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:59:20,596] Trial 379 finished with value: 0.2998836368221046 and parameters: {'boosting_type': 'goss', 'num_leaves': 119, 'max_depth': 7, 'learning_rate': 0.014623764510717004, 'min_split_gain': 0.5089792332474321, 'min_child_weight': 0.021864818185516836, 'min_child_samples': 327, 'colsample_bytree': 0.5448640666539997, 'reg_alpha': 7.986236026022013e-06, 'reg_lambda': 0.006955766410776999, 'colsample_bynode': 0.7933049103187659, 'min_data_per_group': 39, 'max_cat_threshold': 60, 'cat_l2': 90.43671622525697, 'cat_smooth': 0.0017910700379272781, 'max_cat_to_onehot': 4, 'max_bin': 125, 'n_estimators': 422, 'top_rate': 0.22484217139525794, 'other_rate': 0.39041646343869174}. Best is trial 277 with value: 0.29933817748389.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-23 23:59:32,799] Trial 381 finished with value: 0.2999960691425464 and parameters: {'boosting_type': 'goss', 'num_leaves': 111, 'max_depth': 8, 'learning_rate': 0.023116958739668573, 'min_split_gain': 0.03760753422901612, 'min_child_weight': 0.021633498917205292, 'min_child_samples': 259, 'colsample_bytree': 0.6560918299433508, 'reg_alpha': 7.450829752705332e-06, 'reg_lambda': 0.009000414690278324, 'colsample_bynode': 0.7954548078464334, 'min_data_per_group': 36, 'max_cat_threshold': 24, 'cat_l2': 23.050413038504658, 'cat_smooth': 0.0028698934375420747, 'max_cat_to_onehot': 45, 'max_bin': 128, 'n_estimators': 422, 'top_rate': 0.2217880002046127, 'other_rate': 0.39411990797462415}. Best is trial 277 with value: 0.29933817748389.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-24 00:05:18,034] Trial 310 finished with value: 0.3006911836688567 and parameters: {'boosting_type': 'dart', 'num_leaves': 135, 'max_depth': 8, 'learning_rate': 0.016800529834869824, 'min_split_gain': 0.013247824298853409, 'min_child_weight': 0.03921816782702847, 'min_child_samples': 355, 'colsample_bytree': 0.5987365470720236, 'reg_alpha': 2.4355189864473616e-07, 'reg_lambda': 0.0013549745273800584, 'colsample_bynode': 0.6800886514649693, 'min_data_per_group': 8, 'max_cat_threshold': 769, 'cat_l2': 3.6189640198588605e-07, 'cat_smooth': 0.0015112816486029313, 'max_cat_to_onehot': 3, 'max_bin': 316, 'n_estimators': 349, 'subsample': 0.7727014177159663, 'subsample_freq': 2, 'drop_rate': 0.4318835830675534, 'skip_drop': 0.39083504532011903, 'max_drop': 149, 'xgboost_dart_mode': True, 'uniform_drop': False}. Best is trial 277 with value: 0.29933817748389.
[I 2026-04-24 00:06:08,400] Trial 304 finished with value: 0.3006179738265928 and parameters: {'boosting_type': 'dart', 'num_


BEST RMSE: 0.2993
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 146,
    "max_depth": 7,
    "learning_rate": 0.026039278911603174,
    "min_split_gain": 0.06465727022114406,
    "min_child_weight": 0.015060459797037726,
    "min_child_samples": 351,
    "colsample_bytree": 0.3988909719645984,
    "reg_alpha": 4.929485323038812e-06,
    "reg_lambda": 0.006004611468946339,
    "colsample_bynode": 0.7553381328428536,
    "min_data_per_group": 48,
    "max_cat_threshold": 4,
    "cat_l2": 1.4600621139785737e-07,
    "cat_smooth": 5.371832010451595e-07,
    "max_cat_to_onehot": 2,
    "max_bin": 329,
    "n_estimators": 384,
    "top_rate": 0.21903858056830947,
    "other_rate": 0.4333115775153122,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.4140
  colsample_bynode    : 0.2503
  boosting_type       : 0.1695
  colsample_bytree    : 0.0333
  learning_rate       : 0.0278
  n_estimators        : 0.0198
  max_cat_to_onehot   : 0.0179
  num_leaves

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")


# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GOSS: Trees 384 -> 3840, LR 0.0260 -> 0.0026

Optuna Val RMSE:   0.2993
Holdout Test RMSE: 0.2984


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
import lightgbm as lgb

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "goss"]), # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512), # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20), # Max tree depth
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "objective": "regression", # Regression objective
        "metric": "rmse", # Evaluation metric
        "min_split_gain": trial.suggest_float("min_split_gain", 0, 30.0), # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-5, 10.0, log=True), # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500), # Min data in one leaf
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0), # % of features used per tree
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 100.0, log=True), # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 100.0, log=True), # L2 regularization
        "random_state": 42, # Fixed seed for reproducibility
        "n_jobs": 1, # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1, # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.2, 1.0), # % of features used per node
        "min_data_per_group": trial.suggest_int("min_data_per_group", 1, 1000), # Min data per categorical group
        "max_cat_threshold": trial.suggest_int("max_cat_threshold", 1, 1000), # Max splits for categories
        "cat_l2": trial.suggest_float("cat_l2", 1e-8, 100.0, log=True), # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float("cat_smooth", 1e-8, 100.0, log=True), # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 51), # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511), # Max bins for features
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000) # Number of boosting rounds
    }
    
    if params["boosting_type"] == "goss": # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0 # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float("top_rate", 0.05, 0.8) # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float("other_rate", 0.05, 1.0 - params["top_rate"]) # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1.0) # Row subsampling
        params["subsample_freq"] = trial.suggest_int("subsample_freq", 1, 10) # Frequency of subsampling

    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        model = LGBMRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr, 
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_lgbm)
fig1.show()
    
fig2 = plot_param_importances(study_lgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_lgbm, 
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ]
) 
fig3.show()
    
fig1.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/LightGBM/Regression/optuna_lgbm_full_parallel.html")


[I 2026-04-24 00:06:42,279] A new study created in memory with name: no-name-ba4f07c0-dacf-4f1a-b76d-7c5febf08f1a
[I 2026-04-24 00:07:06,733] Trial 1 finished with value: 0.23640058838059355 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 483, 'max_depth': 5, 'learning_rate': 0.13122482984544873, 'min_split_gain': 5.628811926548661, 'min_child_weight': 4.458232224707033e-05, 'min_child_samples': 193, 'colsample_bytree': 0.7502947556145296, 'reg_alpha': 0.03675275220306252, 'reg_lambda': 3.6289764207705226e-05, 'colsample_bynode': 0.8591812981076103, 'min_data_per_group': 278, 'max_cat_threshold': 348, 'cat_l2': 5.29152629673972e-08, 'cat_smooth': 3.6557051723842837e-07, 'max_cat_to_onehot': 14, 'max_bin': 273, 'n_estimators': 397, 'subsample': 0.26349778392042783, 'subsample_freq': 7}. Best is trial 1 with value: 0.23640058838059355.
[I 2026-04-24 00:07:25,865] Trial 6 finished with value: 0.23763938269859142 and parameters: {'boosting_type': 'goss', 'num_leaves': 291, 'max_dep


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-24 01:54:16,152] Trial 124 finished with value: 0.23692173440263373 and parameters: {'boosting_type': 'dart', 'num_leaves': 79, 'max_depth': 11, 'learning_rate': 0.08104506153054891, 'min_split_gain': 23.641615484152663, 'min_child_weight': 0.0005673567349448611, 'min_child_samples': 362, 'colsample_bytree': 0.8527722125339173, 'reg_alpha': 2.4757484348491683e-05, 'reg_lambda': 0.0127839648056485, 'colsample_bynode': 0.7696464056729196, 'min_data_per_group': 941, 'max_cat_threshold': 815, 'cat_l2': 0.2915973466465425, 'cat_smooth': 0.13380677004418295, 'max_cat_to_onehot': 39, 'max_bin': 365, 'n_estimators': 588, 'subsample': 0.8491019071875503, 'subsample_freq': 6, 'drop_rate': 0.11172696140938523, 'skip_drop': 0.36909466321586964, 'max_drop': 67, 'xgboost_dart_mode': True, 'uniform_drop': False}. Best is trial 22 with value: 0.23276763801633027.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-24 01:54:27,645] Trial 125 finished with value: 0.23417823031632853 and parameters: {'boosting_type': 'goss', 'num_leaves': 185, 'max_depth': 10, 'learning_rate': 0.08094516258697602, 'min_split_gain': 1.9174069231764486, 'min_child_weight': 0.00018427256017134543, 'min_child_samples': 368, 'colsample_bytree': 0.6925386545880794, 'reg_alpha': 2.4450064632508477e-05, 'reg_lambda': 2.1898419216334967e-08, 'colsample_bynode': 0.767246137497629, 'min_data_per_group': 997, 'max_cat_threshold': 817, 'cat_l2': 2.0812461594705987e-06, 'cat_smooth': 1.8778404592501235e-07, 'max_cat_to_onehot': 40, 'max_bin': 480, 'n_estimators': 590, 'top_rate': 0.3468977663577621, 'other_rate': 0.4180814692327932}. Best is trial 22 with value: 0.23276763801633027.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-24 01:55:20,490] Trial 126 finished with value: 0.23690056169913068 and parameters: {'boosting_type': 'dart', 'num_leaves': 355, 'max_depth': 12, 'learning_rate': 0.07403668193493983, 'min_split_gain': 23.030189905598895, 'min_child_weight': 0.0012274833717901051, 'min_child_samples': 333, 'colsample_bytree': 0.8615653154244812, 'reg_alpha': 0.0011919633636611662, 'reg_lambda': 1.2277935126165433e-08, 'colsample_bynode': 0.7675900232244138, 'min_data_per_group': 84, 'max_cat_threshold': 556, 'cat_l2': 2.0728449597819245e-06, 'cat_smooth': 92.26801760359825, 'max_cat_to_onehot': 42, 'max_bin': 479, 'n_estimators': 787, 'subsample': 0.8553851909669143, 'subsample_freq': 6, 'drop_rate': 0.10373355808296156, 'skip_drop': 0.3849087212612554, 'max_drop': 64, 'xgboost_dart_mode': True, 'uniform_drop': False}. Best is trial 22 with value: 0.23276763801633027.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-24 01:59:22,643] Trial 5 finished with value: 0.23580288603444588 and parameters: {'boosting_type': 'dart', 'num_leaves': 296, 'max_depth': 20, 'learning_rate': 0.007840476494223488, 'min_split_gain': 2.0641308367169864, 'min_child_weight': 0.033273213939020774, 'min_child_samples': 26, 'colsample_bytree': 0.9803429444604004, 'reg_alpha': 2.477291448056572e-05, 'reg_lambda': 0.2589812998432455, 'colsample_bynode': 0.32184796505536095, 'min_data_per_group': 622, 'max_cat_threshold': 29, 'cat_l2': 37.81367552497411, 'cat_smooth': 0.08828308864505123, 'max_cat_to_onehot': 49, 'max_bin': 457, 'n_estimators': 941, 'subsample': 0.39580132399831114, 'subsample_freq': 6, 'drop_rate': 0.35730386356231825, 'skip_drop': 0.3124753744695168, 'max_drop': 103, 'xgboost_dart_mode': False, 'uniform_drop': True}. Best is trial 22 with value: 0.23276763801633027.
[I 2026-04-24 02:09:39,778] Trial 74 finished with value: 0.23416966070061299 and parameters: {'boosting_type': 'dart', 'num_leaves'


BEST RMSE: 0.2328
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 121,
    "max_depth": 20,
    "learning_rate": 0.04661713210716701,
    "min_split_gain": 0.18290729552773044,
    "min_child_weight": 0.00023524131949363415,
    "min_child_samples": 346,
    "colsample_bytree": 0.5507495684055095,
    "reg_alpha": 0.00027138550155974373,
    "reg_lambda": 0.004348467361754384,
    "colsample_bynode": 0.31991427679720374,
    "min_data_per_group": 800,
    "max_cat_threshold": 719,
    "cat_l2": 6.2826727468026595e-06,
    "cat_smooth": 0.11001706698403341,
    "max_cat_to_onehot": 18,
    "max_bin": 376,
    "n_estimators": 762,
    "top_rate": 0.35233270415254714,
    "other_rate": 0.08126911663902539,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.7243
  boosting_type       : 0.0942
  colsample_bynode    : 0.0345
  max_depth           : 0.0257
  colsample_bytree    : 0.0251
  min_data_per_group  : 0.0203
  num_leaves          : 0.0171
  max

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GOSS: Trees 762 -> 7620, LR 0.0466 -> 0.0047

Optuna Val RMSE:   0.2328
Holdout Test RMSE: 0.2836
